# Meta multi-state posts: Qwen3 32B (prompt v3, no evidence)

Runs only the 4,611 posts in meta_post_analysis that name several US states (T5); single-state posts keep their stored state. Cell 10 writes the answers as two tables keyed by post_id — one row per post, and one row per post and main state with weights — so they join onto meta_post_analysis. Run on an **A100**. Cell 7 checks the answers match the test first. After a disconnect, rerun cells 1–6, 8 and 9 to continue. Documented in docs/Geolocation_Extraction.docx, section 6.

## Cell 1: GPU check and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## Cell 2: install Ollama and start it, allowing 4 posts at once

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time, requests
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_NUM_PARALLEL": "4"})
for _ in range(60):
    try:
        print("Ollama", requests.get("http://localhost:11434/api/version").json()["version"])
        break
    except requests.ConnectionError:
        time.sleep(1)

## Cell 3: download the model (about 20 GB; a few minutes)

In [ ]:
MODEL = "qwen3:32b"
!ollama pull {MODEL}

## Cell 4: connect to the database (same as notebooks/colab_x_corpus.ipynb)

In [ ]:
!pip install -q "cloud-sql-python-connector[pg8000]" sqlalchemy pandas
from google.colab import auth
auth.authenticate_user()     # sign in with your @g.harvard.edu account

import getpass
import pandas as pd
import sqlalchemy
from google.cloud.sql.connector import Connector

INSTANCE = "harvard-capstone-499102:us-central1:h5n1-pg"
DB_PASSWORD = getpass.getpass("h5n1_app password: ")
connector = Connector()

def _getconn():
    return connector.connect(INSTANCE, "pg8000", user="h5n1_app",
                             password=DB_PASSWORD, db="h5n1")

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=_getconn,
                                  pool_pre_ping=True)
print(pd.read_sql("SELECT count(*) FROM meta_post_analysis", engine).iloc[0, 0], "Meta posts")

## Cell 5: the prompt (v3 without evidence)

In [ ]:
# The v3 prompt that produced qwen3_32b_v3.csv, with the evidence answer removed:
# a shorter answer per post means less generation time, so fewer compute units.
STATE_NAME = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT", "delaware": "DE",
    "florida": "FL", "georgia": "GA", "hawaii": "HI", "idaho": "ID",
    "illinois": "IL", "indiana": "IN", "iowa": "IA", "kansas": "KS",
    "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN",
    "mississippi": "MS", "missouri": "MO", "montana": "MT", "nebraska": "NE",
    "nevada": "NV", "new hampshire": "NH", "new jersey": "NJ",
    "new mexico": "NM", "new york": "NY", "north carolina": "NC",
    "north dakota": "ND", "ohio": "OH", "oklahoma": "OK", "oregon": "OR",
    "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA",
    "west virginia": "WV", "wisconsin": "WI", "wyoming": "WY",
    "district of columbia": "DC",
}
SPANISH_STATE = {
    "nueva york": "NY", "carolina del norte": "NC", "carolina del sur": "SC",
    "dakota del norte": "ND", "dakota del sur": "SD", "nuevo mexico": "NM",
    "nuevo méxico": "NM", "luisiana": "LA", "pensilvania": "PA",
    "virginia occidental": "WV", "virginia del oeste": "WV",
    "nueva jersey": "NJ", "nuevo hampshire": "NH", "misisipi": "MS",
    "misisipí": "MS", "misuri": "MO", "hawái": "HI",
    "nueva inglaterra": None,
    "tejas": "TX", "georgia": "GA",
}
CODES = sorted(set(STATE_NAME.values()))
_es = {}
for name, code in SPANISH_STATE.items():
    if code:
        _es.setdefault(code, []).append(name)
REFERENCE = "\n".join(
    f"{code} = {en.title()}" + (f" (es: {', '.join(n.title() for n in _es[code])})"
                                if code in _es else "")
    for en, code in sorted(STATE_NAME.items(), key=lambda kv: kv[1]))

SYSTEM = f"""You read social-media posts about avian influenza (bird flu). Posts are in English or Spanish.

Task: find the US states where the post reports a bird-flu EVENT. An event is a detection in birds or animals, a human case, an outbreak, a cull, a quarantine, or a state's own response such as an emergency declaration.

Rules:
- Use only codes from the reference list below.
- A post can report events in several states. List every one.
- Do NOT count a state that is only part of another place's name: Lake Michigan, Kansas City, Washington County (Oregon), Georgia del Sur / South Georgia, Baja California, Sierra Nevada, La Florida (Chile).
- Do NOT count a state name that is part of the name of a species, breed, product, company or publication: California condor, Carolina wren, Mississippi kite, Rhode Island Red (a chicken breed), Jersey cows, Kentucky Fried Chicken, The New York Times, The Washington Post. Place the event where it happened: a California condor that died in Arizona is an Arizona event.
- Do NOT count "Washington" when it means the US federal government.
- Do NOT count a state named only for contrast, history or comparison ("unlike Iowa, we have no cases").
- Places outside the US are not states. If the event is outside the US, return no states.
- If the post is about the US as a whole with no specific state, return no states.

states: every US state with an event, or [] if none.
main_states: the state or states the post is mainly about, judged from context. Usually this is one state: a post about a sick bird at the Oregon Zoo in Portland is ["OR"], even if it mentions other states in passing. When the post gives several states equal weight -- for example, dairy herds confirmed infected in Texas, Kansas and Michigan -- list all of them. Every code in main_states must also be in states. Use [] when states is empty.
us_event: true if the post reports a specific bird-flu event inside the US, else false.

Reference list of US states:
{REFERENCE}"""

SCHEMA = {
    "type": "object",
    "properties": {
        "states": {"type": "array", "items": {"type": "string", "enum": CODES}},
        "main_states": {"type": "array", "items": {"type": "string", "enum": CODES}},
        "us_event": {"type": "boolean"},
    },
    "required": ["states", "main_states", "us_event"],
}

## Cell 6: the model call

In [ ]:
import json
from pathlib import Path

PROMPT_VERSION = "v3_noevidence"
MAX_CHARS = 2000          # same as the test: the model reads the first 2,000 characters
USE_HINT = True           # same as the test: the keyword search's states, marked "may be wrong"
OUT = Path("/content/drive/MyDrive/h5n1_llm_pilot")

def ask(post, hint):
    user = post[:MAX_CHARS]
    if hint:
        user += f"\n\n(A keyword search found these state names in the text: {hint}. They may be wrong.)"
    body = {"model": MODEL, "stream": False, "format": SCHEMA, "think": False,
            "options": {"temperature": 0, "num_ctx": 4096,
                        "num_predict": 128},   # the answer is ~30 tokens; caps a runaway reply
            "messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": user}]}
    r = requests.post("http://localhost:11434/api/chat", json=body, timeout=600)
    r.raise_for_status()
    return json.loads(r.json()["message"]["content"])

def run_one(p):
    """One post -> one result row. Errors are recorded, not raised, so the run keeps going."""
    try:
        a = ask(p.post_text, p.geo_matched if USE_HINT else None)
        main = set(a.get("main_states", []))
        states = set(a.get("states", [])) | main      # main states are always among the states
        return {"post_id": str(p.post_id), "llm_states": ",".join(sorted(states)),
                "llm_main": ",".join(sorted(main)), "llm_event": a.get("us_event"), "error": ""}
    except Exception as e:
        return {"post_id": str(p.post_id), "llm_states": "", "llm_main": "", "llm_event": None,
                "error": str(e)[:200]}

## Cell 7: check: does dropping evidence change the answers? (50 labelled posts, a few minutes)

In [ ]:
# The labelled sample is all multi-state (T5) posts -- exactly the posts the LLM will see.
# Reruns 50 posts of the 200-post sample and compares with qwen3_32b_v3.csv from the test.
sample = pd.read_csv(OUT / "sample.csv", dtype=str, keep_default_na=False).head(50)
old = pd.read_csv(OUT / "qwen3_32b_v3.csv", dtype=str, keep_default_na=False)
t0 = time.time()
check = pd.DataFrame([run_one(p) for p in sample.itertuples()])
print(f"{(time.time() - t0) / len(sample):.1f} s per post, one at a time")
cmp = check.merge(old[["post_id", "llm_states", "llm_main"]], on="post_id", suffixes=("", "_test"))
print("same state list:", f"{(cmp.llm_states == cmp.llm_states_test).mean():.0%}",
      "| same main states:", f"{(cmp.llm_main == cmp.llm_main_test).mean():.0%}", f"(n={len(cmp)})")
cmp[(cmp.llm_states != cmp.llm_states_test) | (cmp.llm_main != cmp.llm_main_test)]

## Cell 8: the posts to run: every multi-state post (T5) in meta_post_analysis

In [ ]:
# Posts naming one state (T1-T4) keep the state already stored; only T5 needs deciding.
posts = pd.read_sql(
    "SELECT post_id, geo_tier, geo_matched, post_text FROM meta_post_analysis "
    "WHERE geo_tier = 'T5_multi' ORDER BY post_id", engine)
posts["post_id"] = posts.post_id.astype(str)
print(f"{len(posts):,} multi-state posts to label")

## Cell 9: the LLM run: 4 posts at a time, saved to Drive every 200, resumes after a drop

In [ ]:
from concurrent.futures import ThreadPoolExecutor

TAG = f"{MODEL.replace(':', '_')}_{PROMPT_VERSION}"
raw_path = OUT / f"meta_T5_{TAG}.csv"
COLS = ["post_id", "geo_matched", "llm_states", "llm_main", "llm_event", "error"]
done = (pd.read_csv(raw_path, dtype=str, keep_default_na=False)
        if raw_path.exists() else pd.DataFrame(columns=COLS))
done = done[done.error == ""]                       # failed posts are tried again
todo = posts[~posts.post_id.isin(set(done.post_id))]
print(f"{len(done):,} done, {len(todo):,} to go")

t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    for start in range(0, len(todo), 200):
        chunk = todo.iloc[start:start + 200]
        new = pd.DataFrame(list(pool.map(run_one, chunk.itertuples())))
        new = new.merge(chunk[["post_id", "geo_matched"]], on="post_id")
        done = pd.concat([done, new.astype(str)], ignore_index=True)[COLS]
        done.to_csv(raw_path, index=False)
        rate = (start + len(chunk)) / (time.time() - t0)
        left = (len(todo) - start - len(chunk)) / rate / 3600
        print(f"  {len(done):,}/{len(posts):,}  {rate:.2f} posts/s  ~{left:.1f} h left")
print(f"wrote {raw_path.name}: {len(done):,} rows, {(done.error != '').sum()} errors")

## Cell 10: shape the answers so they can be added to meta_post_analysis

In [ ]:
# A post can have several main states, so one `state` column cannot hold the answer.
# Two tables, both keyed by post_id:
#   wide -- one row per post: lists as comma-separated text, as meta_post_analysis
#           already stores search_terms. Joins one-to-one onto meta_post_analysis.
#   long -- one row per post and main state, weighted 1/n, so a post about Kansas and
#           Texas counts 0.5 for each when posts are totalled by state.
raw = pd.read_csv(raw_path, dtype=str, keep_default_na=False)
assert raw.post_id.is_unique and set(raw.post_id) == set(posts.post_id), \
    "some T5 posts are missing or failed: rerun cell 9 until it reports 0 to go and 0 errors"

wide = pd.DataFrame({
    "post_id": raw.post_id,
    "llm_states": raw.llm_states,                       # every state with an event, e.g. "KS,MI,TX"
    "llm_main_states": raw.llm_main,                    # the state(s) the post is mainly about
    "n_main_states": raw.llm_main.map(lambda s: len([c for c in s.split(",") if c])),
    "llm_us_event": raw.llm_event.map({"True": True, "False": False}),
    "llm_model": TAG,
})
long = (wide[wide.n_main_states > 0]
        .assign(state=wide.llm_main_states.str.split(","))
        .explode("state")[["post_id", "state", "n_main_states"]])
long["weight"] = 1 / long.n_main_states

wide.to_csv(OUT / f"meta_T5_states_wide_{TAG}.csv", index=False)
long.to_csv(OUT / f"meta_T5_states_long_{TAG}.csv", index=False)

print(f"wide: {len(wide):,} posts | with a main state: {(wide.n_main_states > 0).sum():,} | "
      f"with several: {(wide.n_main_states > 1).sum():,} | no US event state: {(wide.n_main_states == 0).sum():,}")
print(f"long: {len(long):,} rows | weights sum to {long.weight.sum():,.0f} "
      f"(= posts with a main state)")
print(long.groupby("state").weight.sum().sort_values(ascending=False).head(15).round(1))

## Cell 11: check that the answers join onto meta_post_analysis

In [ ]:
meta = pd.read_sql("SELECT post_id, geo_tier, state FROM meta_post_analysis", engine)
meta["post_id"] = meta.post_id.astype(str)
joined = meta.merge(wide, on="post_id", how="left", validate="one_to_one")
t5 = joined[joined.geo_tier == "T5_multi"]
print("rows before and after the join:", len(meta), len(joined))
print("T5 posts with an LLM answer:", t5.llm_model.notna().sum(), "of", len(t5))
print("non-T5 posts given an LLM answer by mistake:", joined[joined.geo_tier != "T5_multi"].llm_model.notna().sum())

# To add to the database, write the two tables beside meta_post_analysis rather than
# altering it (build_meta_analysis.py replaces that table on every rebuild), then join:
#   wide.to_sql("meta_post_t5_state", engine, if_exists="replace", index=False, chunksize=1000)
#   long.to_sql("meta_post_t5_state_long", engine, if_exists="replace", index=False, chunksize=1000)
#   SELECT m.*, t.llm_states, t.llm_main_states, t.n_main_states
#   FROM meta_post_analysis m LEFT JOIN meta_post_t5_state t USING (post_id);